# Post Generator

In [10]:
# Tweet Style Generator (Keras LSTM)

import pandas as pd
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import random

class PostGenerator:
    def __init__(self, csv_path, seq_length=5, embed_dim=50):
        self.seq_length = seq_length
        self.embed_dim = embed_dim

        # Load and clean tweets
        df = pd.read_csv(csv_path)
        tweets = df['content'].dropna().astype(str).tolist()
        self.cleaned = [self.clean_text(t) for t in tweets]
        text = " ".join(self.cleaned)

        # Tokenize
        self.tokenizer = Tokenizer()
        self.tokenizer.fit_on_texts([text])
        seq = self.tokenizer.texts_to_sequences([text])[0]
        self.vocab_size = len(self.tokenizer.word_index) + 1

        # Create sequences
        sequences = []
        for i in range(seq_length, len(seq)):
            sequences.append(seq[i-seq_length:i+1])
        sequences = np.array(sequences)
        X, y = sequences[:, :-1], sequences[:, -1]
        y = to_categorical(y, num_classes=self.vocab_size)

        # Define and compile model
        self.model = Sequential([
            Embedding(input_dim=self.vocab_size, output_dim=embed_dim),
            LSTM(128, return_sequences=True),
            LSTM(128),
            Dense(self.vocab_size, activation='softmax')
        ])
        self.model.compile(loss='mse', optimizer='adam', metrics=['accuracy'])
        self.model.summary()

        # Train model
        self.model.fit(X, y, batch_size=128, epochs=20, verbose=1)

    def clean_text(self, text):
        text = text.lower()
        text = re.sub(r"http\S+|@\S+|#[\S]+", "", text)  # remove links, mentions, hashtags
        text = re.sub(r"[^a-z\s]", "", text)  # remove punctuation
        return text.strip()

    def sample_with_temperature(self, preds, temperature=1.0):
        preds = np.asarray(preds).astype('float64')
        preds = np.log(preds + 1e-8) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)
        return np.random.choice(len(preds), p=preds)

    def generate_post(self, seed_text=None, char_limit=240, temperature=0.7):
        if seed_text is None:
            seed_text = random.choice(self.cleaned).split()[:self.seq_length]
            seed_text = " ".join(seed_text)

        result = []
        current_text = seed_text.strip()
        target_length = random.randint(80, char_limit)
        recent_words = []

        while len(current_text) < target_length:
            encoded = self.tokenizer.texts_to_sequences([current_text])[0]
            encoded = encoded[-self.seq_length:]
            padded = np.pad(encoded, (self.seq_length - len(encoded), 0))
            preds = self.model.predict(padded.reshape(1, -1), verbose=0)
            pred_idx = self.sample_with_temperature(preds[0], temperature)
            next_word = self.tokenizer.index_word.get(pred_idx, '')

            if not next_word:
                break

            current_text += ' ' + next_word
            result.append(next_word)

            recent_words.append(next_word)
            if len(recent_words) > 3:
                recent_words.pop(0)
                if recent_words[0] == recent_words[1] == recent_words[2]:
                    break

        return current_text

# --- Example Use ---
# generator = TweetGenerator("your_tweets.csv")
# print(generator.generate_post())


In [11]:
generator = PostGenerator('post_content.csv')
new_post = generator.generate_post()
print(new_post)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 9.3332e-04 - loss: 6.5660e-04
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.0239 - loss: 6.5660e-04
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.0407 - loss: 6.5660e-04
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.0399 - loss: 6.5660e-04
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.0423 - loss: 6.5660e-04
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.0413 - loss: 6.5660e-04
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.0411 - loss: 6.5659e-04
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.0420 - loss: 6.5659e-04
Epoch 9/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.0389 - loss: 6.5659e-04
Epoch 10/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.0429 - loss: 6.5659e-04
Epoch 11/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.0429 - loss: 6.5659e-04
Epoch 12/20
35/

In [17]:
generator.generate_post()

'the end is nigh creating gc charcuterie mice confused models enhancements biasvariance'

In [18]:
generator.generate_post()

'birds eye view finish what shortcut recommendation sessions localized traffic ill pretty two seesv fair season duuuuuuuuude reminds'

In [21]:
generator.generate_post()

'my third artist is now heavy excuses setting have like participate chance portfolios users'

In [22]:
generator.generate_post()

'thanks for keeping me honest brokers axisaxes approaches selected old higher sense maintenance hilarious absolutely all bananas sincerely formula simulation through like builder enjoy json glance damn hardware'

In [23]:
generator.generate_post()

' bro im palkia disclaimer intents solstice named hot with ice deadpool leverage spent suddenly'

In [31]:
for _ in range(10):
    post = generator.generate_post()
    print("- ", post, "\n")

-  have fun scratch inspiration pay fast c increase regret var companion alive let mail mostly hyperparameter policy nutshell effectively moves field using tend stage checkpoints widgets but die 

-  make technology tactile again head ethics away dbs approach beyond listening claude finally editing dividends spotlight tried auth smiling audio tweeting beget yours maybe video everywhere 

-  downloaded my x analytics to stack vibe keeping abstracted urge colon friends labor the wont people made prescriptive cuisines interface swiftui spotify var needed create froggy increase 

-  deadpool amp wolverine is perfect thread topic closer basics later milestonetrained refactor promise keynote playlist han album explains opportunity mainlining declarative gc mercayyy mvs onhover open paper faith 

-  solarpowered zen is the theme antipatterns tool same descriptions feeding overall solve concerns please storing 

-  has anyone built a tool cool the hole spent glass whip kiss used ethics automat